<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Basic HTTP client use

**[Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/)** · Python for Practical AI Engineering · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** get ticket records from a web service with httpx, read status codes, decode JSON, always set a timeout, and test the error cases with mocked responses instead of the network.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection: the setup installs httpx and downloads the data, and the lesson downloads one small public file. No account and no cost. |
| **You should know** | `read_rows`, `split_records` and `build_summary`, from [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here, and installs httpx. |
| **Tested** | Python 3.14.6 with httpx 0.28.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M04-L02-basic-http-client-use/basic-http-client-use-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1
import httpx; print("httpx", httpx.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 9 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
from pathlib import Path

from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.report import build_summary

INPUT_FILE = Path("data/tickets.csv")
OUTPUT_FILE = Path("reports/summary.json")


def main() -> None:
    rows = read_rows(INPUT_FILE)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `fresh("ticket_cleaner.parsing")` imports a module of the project again, so that the notebook uses the newest version of its file. (A plain `import` uses the copy that is already in memory: see [Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/).)
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

import importlib

def fresh(module_name):
    """Import a module of the project again, from its newest file."""
    for name in list(sys.modules):
        if name == "ticket_cleaner" or name.startswith("ticket_cleaner."):
            del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module(module_name)

def check_mocks():
    """Call fetch_rows with the two mock functions, and check the results."""
    import httpx
    remote = fresh("ticket_cleaner.remote")
    url = "https://tickets.example/api/tickets"
    for answer in [answer_unauthorized, answer_bad_rows]:
        if not isinstance(answer(httpx.Request("GET", url)), httpx.Response):
            print(f"Not yet: {answer.__name__} must return an httpx.Response.")
            return
    try:
        remote.fetch_rows(url, client=httpx.Client(transport=httpx.MockTransport(answer_unauthorized)))
        print("Not yet: answer_unauthorized must give a response that fetch_rows rejects. Check the status code.")
        return
    except httpx.HTTPStatusError as error:
        if error.response.status_code != 401 or error.response.text != "missing token":
            print(f"Not yet: answer_unauthorized gives {error.response.status_code} {error.response.text!r}. The task asks for 401 with the text 'missing token'.")
            return
    except Exception as error:
        print(f"Not yet: answer_unauthorized must return an httpx.Response ({type(error).__name__}: {error}).")
        return
    try:
        rows = remote.fetch_rows(url, client=httpx.Client(transport=httpx.MockTransport(answer_bad_rows)))
    except Exception as error:
        print(f"Not yet: answer_bad_rows must return status 200 with a JSON list ({type(error).__name__}: {error}).")
        return
    if rows != [{"id": "T-2002", "status": "opne"}]:
        print(f"Not yet: answer_bad_rows gives {rows!r}. Check the JSON list in the task.")
        return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Request and response

When your program gets data from a web service, it is the **client**, and the service is the **server**. They talk with **HTTP**. A **request** has a method (`GET` means "send me this resource"), a URL and headers. A **response** has a **status code**, headers and a **body**.

The status code says how the request went: `2xx` is success, `4xx` is a problem with the request (`404` Not Found, `401` Unauthorized), and `5xx` is a problem on the server (`503` Service Unavailable). Check the code before you use the body.

## 2. Worked example: get the tickets

The labs repository publishes the same 15 tickets as JSON at a public URL. `timeout=10.0` means: if the server does not answer within 10 seconds, stop waiting and raise an error.

> **Predict.** How many rows does the response have? Then run the cell.

In [ ]:
import httpx

url = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"
response = httpx.get(url, timeout=10.0)
print(response.status_code)
print(response.headers["content-type"])
rows = response.json()
print(len(rows))
print(rows[0])

> **Check.** You should see `200`, `text/plain; charset=utf-8`, `15` and the first ticket as a dictionary. `response.json()` decodes the JSON body, as `json.load` does for a file. (This server says `text/plain` for every file. A real web service says `application/json`.)

Now ask for a file that does not exist. `raise_for_status()` does nothing for a `2xx` code, and raises an error for a `4xx` or `5xx` code. Run the cell.

In [ ]:
missing = httpx.get(url.replace("tickets.json", "nothing.json"), timeout=10.0)
print(missing.status_code)
print(missing.text)
try:
    missing.raise_for_status()
except httpx.HTTPStatusError as error:
    print("HTTPStatusError:", error)

> **Check.** You should see `404`, `404: Not Found`, then `HTTPStatusError: Client error '404 Not Found' for url '…/C02/data/nothing.json'` and a link to an explanation of the code. Call `raise_for_status()` before you read the body, so that a wrong URL stops the program with a clear message.

## 3. The fetch function

`fetch_rows` returns a list of raw records, like `read_rows`, so the rest of the program does not care where they came from. The caller can give a ready-made `client`. Normal code does not, and gets a new one with a 10-second timeout. `with client:` closes the network connections at the end. Run the cell.

In [ ]:
%%writefile ticket_cleaner/remote.py
import httpx


def fetch_rows(url: str, client: httpx.Client | None = None) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url)
        response.raise_for_status()
        return response.json()

## 4. Test without the network: mocked responses

A **mock** is a stand-in for a real part, which you control. `httpx.MockTransport` makes a client that does not use the network: it calls your function to make each response. The domain `tickets.example` does not even exist.

Run the next two cells. The first saves `try_mock.py`. The second runs it.

In [ ]:
%%writefile try_mock.py
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()

In [ ]:
run("try_mock.py")

> **Check.** You should see `[{'id': 'T-2001', 'status': 'open'}]`, then a traceback that ends with `httpx.HTTPStatusError: Server error '503 Service Unavailable' for url 'https://tickets.example/api/tickets'`, and `(exit code 1)`. No request left the computer. The second call stops with a clear error, before any bad data reaches `split_records`.

## 5. Guided practice: a second source

`count_tickets.py` gets a choice of source: a file path, or `"remote"`. Run the cell, then the `run` cell.

In [ ]:
%%writefile count_tickets.py
from pathlib import Path

from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

SOURCE = "remote"  # a file path, or "remote"
TICKETS_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"
OUTPUT_FILE = Path("reports/summary.json")


def load_rows(source: str) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if source == "remote":
        return fetch_rows(TICKETS_URL)
    return read_rows(Path(source))


def main() -> None:
    rows = load_rows(SOURCE)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
run("count_tickets.py")

> **Check.** You should see the same output as for the CSV file: six `Row … rejected` lines, `15 rows: 9 valid, 6 rejected.` and `Report: reports/summary.json`.

> **Your turn: change one thing.** In the cell above, change `SOURCE` to `"data/tickets.csv"`, and run both cells again. The result is the same.

## 6. Your turn: test a 401 and a 200 with bad data

> **Your turn.** Write two mock functions in the next cell:
>
> 1. `answer_unauthorized`: status `401`, with the text `"missing token"`.
> 2. `answer_bad_rows`: status `200`, with the JSON list `[{"id": "T-2002", "status": "opne"}]`.
>
> Predict what `fetch_rows` does with each one. Run the cell, then the cell after it, which calls `fetch_rows` with each function. Then run the check cell.

In [ ]:
import httpx


def answer_unauthorized(request: httpx.Request) -> httpx.Response:
    return httpx.Response(401, text="missing token")


def answer_bad_rows(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2002", "status": "opne"}])

In [ ]:
remote = fresh("ticket_cleaner.remote")
for answer in [answer_unauthorized, answer_bad_rows]:
    if not isinstance(answer(httpx.Request("GET", "https://tickets.example/api/tickets")), httpx.Response):
        print(answer.__name__, "does not return an httpx.Response yet.")
        continue
    client = httpx.Client(transport=httpx.MockTransport(answer))
    try:
        print(answer.__name__, "returned", remote.fetch_rows("https://tickets.example/api/tickets", client=client))
    except Exception as error:
        print(answer.__name__, "raised", type(error).__name__ + ":", error)

In [ ]:
check_mocks()

`fetch_rows` checks only that the request worked. It does not check the records: that is the job of `split_records`. Run the cell. It checks the record from `answer_bad_rows`.

In [ ]:
parsing = fresh("ticket_cleaner.parsing")
tickets, rejected = parsing.split_records([{"id": "T-2002", "status": "opne"}])
print(rejected)

> **Check.** You should see one `Rejected` record for row 1, with `missing category`, `missing priority` and `unknown status 'opne'`. Each function has one job.

## 7. Failure case: a JSON error from a missing page

Tomás reads the body without a check of the status code.

> **Predict.** What error do you get for the missing file? Then run the cell.

In [ ]:
import json

missing = httpx.get(url.replace("tickets.json", "nothing.json"), timeout=10.0)
try:
    missing.json()
except json.JSONDecodeError as error:
    print("JSONDecodeError:", error)

> **Check.** You should see `JSONDecodeError: Extra data: line 1 column 4 (char 3)`. The body is the error page `404: Not Found`, not JSON. The message talks about JSON, so it hides the real problem.

**Fix:** call `response.raise_for_status()` before `response.json()`. The error then names the real problem: `404 Not Found` and the URL. Check the URL, and open it in a browser to see what the server sends.

> **Warning.** `httpx.ConnectTimeout` means that the server did not answer within the timeout. `httpx.ConnectError` means that the computer cannot find the server's name: check the URL and your internet connection. Both come from `client.get`, before any status code exists.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/). They count toward your certificate when you are signed in and enrolled. On your computer, make `remote.py` and `try_mock.py` in your project, as in the guided practice of the lesson.

In this lesson you learned that a response has a status code, headers and a body, that `raise_for_status()` must come before `response.json()`, that every request needs a timeout, and that a function that accepts a client can be tested with `MockTransport`, without the network.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.